# 第07课：函数进阶

本笔记本是课堂讲义。每个知识点包含：理论知识、案例代码、讲解、易错点与练习。综合练习 P1 使用同目录的 [mobile_game_info.csv](mobile_game_info.csv)。课后独立练习见 [chapter07_函数进阶_课后练习.ipynb](chapter07_函数进阶_课后练习.ipynb)，数据为 [googleplaystore.csv](googleplaystore.csv)。

## 学习目标

1. 不把自定义函数或变量命名为 `print`、`list`、`str` 等内置名。
2. 使用默认参数，使少传一个参数时函数仍能运行。
3. 一次 `return` 多个值，说明返回类型是元组 `tuple`。
4. 用 `a, b = fn()` 拆包两个返回值。
5. 指出函数内部的赋值不会改掉外面的同名变量；本课不要使用 `global`。

## 学习知识点

| 命名 | 默认与多返回 | 作用域 |
| --- | --- | --- |
| 不要覆盖 `print` | 默认参数少传也能跑 | 函数内是局部盒子 |
| 不要占用 `list` / `str` | `return a, b` 是元组 | 外面的同名变量仍在 |
| 覆盖后要恢复内置 | `a, b = fn()` 拆包 | 不用 `global` |

## 基础回顾与案例提问

第6课已经会 `def`、参数、`return` 以及抽列 / 频数。本课补默认参数、多返回值，以及“里面的 x 不是外面的 x”。

1. **R.1** 若把自定义函数命名为 `print`，再写 `print("hello")`，还会不会调用原来的打印？怎样才能再次使用真正的打印？
2. **R.2** `return a + b, a - b` 一次交回几个值？`type(...)` 会显示什么？
3. **R.3** 外面 `x = 10`，函数里 `x = 1`。调用结束后，外面的 `x` 是 10 还是 1？

**作答：** 预测：____；依据：____；验证后说明：____。

所有游戏案例均取自 [mobile_game_info.csv](mobile_game_info.csv)。使用 Python 3；只需标准库 `csv`。从该文件夹启动内核。本课 CSV 可能带有 UTF-8 BOM，使用 `encoding="utf-8-sig"`。本课不讲闭包；不要使用 `global` 关键字。不要使用列表推导式、pandas 或 `Counter`。


In [ ]:
# R.1–R.3: Write and verify your predictions here.


## 1. 不要用内置名当函数名

### 理论知识

**`print`、`list`、`str`、`type` 已经是 Python 准备好的工具名。** 若再写 `def print(...)`，这个名字就被你的函数占用。之后普通的 `print("hello")` 不再指向原来的打印，参数个数也可能对不上，后面的格子会一起坏掉。

本课允许做一次受控演示：先把真正的打印存进 `builtin_print`，再定义同名函数，演示结束后必须恢复。不要在后面的格子里继续使用被覆盖的 `print`。

故障定义（理解即可；真正运行见下一格的受控写法）：

```python
# Deliberate fault if you never save or restore the builtin.
def print(msg):
    pass
print("hello")
```

### 案例：先保存内置 `print`，演示覆盖，再恢复


In [1]:
builtin_print = print

def print(msg):
    builtin_print("被覆盖的 print 收到:", msg)

print("hello")
builtin_print("调用真正的打印函数才恢复输出")
print = builtin_print
print("restored builtin print:", 1 + 1)


被覆盖的 print 收到: hello
调用真正的打印函数才恢复输出
restored builtin print: 2


### 讲解

第一行把真正的打印存起来。自定义 `print("hello")` 只会走到你写的函数体，因此看到“被覆盖的 print 收到: hello”。后面的格子还需要正常 `print`，所以本格末尾执行 `print = builtin_print` 恢复。若内核里残留被覆盖的 `print`，重启内核。

### 易错点与练习

覆盖发生在这个内核的名字表里，不是只影响一个单元格。恢复或重启之前，所有 `print(a, b)` 这类多参数调用都可能失败。

1. **K1.1** 为什么要在 `def print` **之前**写 `builtin_print = print`？若写在定义之后，存下来的是谁？
2. **K1.2** 把抽列函数命名为 `list` 会有什么麻烦？取出一列后又写 `list(...)` 时还是原来的转换吗？

**作答：** 保存顺序：____；函数名叫 `list` 的后果：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 2. 默认参数：少传一个也能跑

### 理论知识与语法

**默认参数给某个参数预备一个常用值。** 调用时可以不传这一项。

```python
def add_sub(a, b=1):
    return a + b, a - b
```

`add_sub(8)` 相当于 `b` 使用 1；`add_sub(8, 3)` 则用你传入的 3。有默认值的参数要写在没有默认值的参数后面。

### 案例：`add_sub(8)` 与 `add_sub(8, 3)`


In [2]:
def add_sub(a, b=1):
    return a + b, a - b

both = add_sub(8)
print("add_sub(8) =", both)
print("add_sub(8, 3) =", add_sub(8, 3))


add_sub(8) = (9, 7)
add_sub(8, 3) = (11, 5)


### 讲解

`add_sub(8)` 得到 `(9, 7)`：8+1 与 8-1。`add_sub(8, 3)` 得到 `(11, 5)`。默认值只在“这一次没传”时使用，不会阻止你传入别的数。

抽列时也可以让 `index` 默认指向评分列 2，少传一个下标就抽评分；需要名字列时再显式传 1。

### 易错点与练习

`def add_sub(a=1, b):` 语法不成立：没有默认值的 `b` 不能排在有默认值的 `a` 后面。

1. **K2.1** 预测 `add_sub(10)` 与 `add_sub(10, 2)` 的两个返回值。
2. **K2.2** 若抽列函数写成 `def extract_column(table, index=2):`，`extract_column(body)` 抽出的是哪一列？要抽类型列应怎样调用？

**作答：** 两组返回值：____；默认抽列与显式下标：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 3. 一次返回多个值，类型是元组

### 理论知识

**逗号分开的多个返回值，类型是 `tuple`（元组）。** 元组可以理解成“暂时捆在一起、按位置取”的一排结果。`both[0]` 是第一个，`both[1]` 是第二个。下一节再学拆包。

`return a + b, a - b` 与 `return (a + b, a - b)` 在本课效果相同：都交回一个含两项的元组。

### 案例：查看 `type`


In [3]:
both = add_sub(8)
print(both)
print(type(both))
print(both[0], both[1])


(9, 7)
<class 'tuple'>
9 7


### 讲解

打印 `both` 会看到圆括号包起来的两个数 `(9, 7)`。`type(both)` 显示 `tuple`。下标 0 是和，下标 1 是差。不要把元组误认为“两个并列的独立变量已经自动诞生”——在拆包之前，它们还捆在一个对象里。

### 易错点与练习

`print(add_sub(8))` 能看到两个数，不等于你已经有了名叫 `s` 和 `d` 的变量。需要两个名字时用拆包。

1. **K3.1** `type(add_sub(8, 3))` 预测是什么？
2. **K3.2** `add_sub(8)[0]` 是 9 还是 7？和与差的顺序由 `return` 的书写顺序决定。

**作答：** 类型：____；`[0]` 是哪一个：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 4. 拆包：`a, b = fn()`

### 理论知识

**左边写两个名字，右边是返回两个值的调用，按顺序解开。** `s, d = add_sub(8, 3)` 之后，`s` 是 11，`d` 是 5。

左右两边个数必须一致。只写一个名字就接到整个元组；写三个名字会出错。

### 案例：拆开和与差


In [4]:
s, d = add_sub(8, 3)
print("unpacked s =", s)
print("unpacked d =", d)
print("s + d =", s + d)


unpacked s = 11
unpacked d = 5
s + d = 16


### 讲解

11 与 5 分别进入 `s` 与 `d`。`s + d` 等于 16，也等于原来的 `8 + 8`，可作快速核对。验收要求：能拆包两个返回值，而不是只会打印整个元组。

### 易错点与练习

`s = d = add_sub(8, 3)` 不是拆包，而是两个名字都指向同一个元组。拆包必须写成 `s, d = ...`。

1. **K4.1** 写出拆包 `add_sub(8)` 的一行代码，并预测两个变量的值。
2. **K4.2** 若写成 `s, d, extra = add_sub(8, 3)`，会怎样？

**作答：** 默认参数拆包结果：____；三个名字接两个值：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 5. 默认下标抽列：读文件后少传一个参数

### 理论知识

把第6课的抽列加上默认 `index=2`（完整文件的评分列）。`extract_column(body)` 抽评分；`extract_column(body, 1)` 抽名字。默认值必须符合**当前这张表**的表头，课后文件不能照抄 2。

### 案例：读取后用默认参数抽列


In [5]:
import csv

with open("mobile_game_info.csv", "r", encoding="utf-8-sig", newline="") as file:
    rows = list(csv.reader(file))
header = rows[0]
body = rows[1:]
print("Score index:", header.index("score"))
print("Title index:", header.index("title"))

def extract_column(table, index=2):
    column = []
    for row in table:
        column.append(row[index])
    return column

scores = extract_column(body)
titles = extract_column(body, 1)
print(titles[0], titles[1], titles[2])
print(scores[0], scores[1], scores[2])


Score index: 2
Title index: 1
人格解体 一念逍遥 明日方舟
8.8 6.5 8.2


### 讲解

默认抽到评分列前三个文本 8.8、6.5、8.2；显式传 1 得到人格解体、一念逍遥、明日方舟。函数名仍叫 `extract_column`，不要叫 `list`。不要一次打印整列。

### 易错点与练习

课后 `googleplaystore.csv` 的 Rating 碰巧也是下标 2，这是两份表各自表头的结果，不是“所有表的第 2 列都是评分”。每次都要 `header.index`。

1. **K5.1** `extract_column(body)` 与 `extract_column(body, 2)` 是否同一列？
2. **K5.2** 要默认抽类型列，应把默认值改成几？根据本课表头回答。

**作答：** 两种调用是否相同：____；类型列默认值：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 6. 全局盒子与局部盒子

### 理论知识

**函数外面的变量和函数里面赋值得到的同名变量，不是同一格。** 外面的属于模块（本笔记本内核）这一层；函数里赋值的 `x` 是局部的，函数结束就不再供外面使用。

读取外面已经存在的变量（只读、不赋值）在简单例子里可能看到外面的值；**一旦在函数里对这个名字赋值，这个名字在函数体内就是局部的。** 本课不展开更细的规则，也不使用 `global`。需要把结果带出去，用 `return`。

### 案例：两个同名 `x`


In [6]:
x = 10

def show_outer():
    print("read x inside function:", x)

show_outer()
print("outside still:", x)


read x inside function: 10
outside still: 10


### 讲解

`show_outer` 内部没有给 `x` 赋值，打印的是外面的 10。外面的 `x` 仍是 10。下一节演示函数内赋值。

### 易错点与练习

不要靠“函数里直接改外面的变量”传递结果。一眼看不出数据从哪来、改了谁。用参数传入，用 `return` 传出。

1. **K6.1** `show_outer` 有没有 `return`？它怎样把信息送到屏幕，又有没有改外面的 `x`？
2. **K6.2** 若需要函数计算出新的 `x` 给后面用，应 `return` 还是在函数里写 `x = 1`？

**作答：** 只读外面的 x：____；把新值带出去的正确做法：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 7. 函数内 `x = 1`，外面仍是 10

### 理论知识

这是本课验收口答：**里面赋值，外面的同名变量不变。**

```python
x = 10

def change():
    x = 1
    return x
```

`change()` 返回 1；调用结束后外面的 `x` 仍是 10。函数里的 `x = 1` 新建了局部变量，没有改写外面那一格。

不要写 `global x` 去强行改外面。本课的纪律是：需要外面用结果，就 `return`。

### 案例：对照返回值与外面的 x


In [7]:
x = 10

def change():
    x = 1
    return x

inner = change()
print("outside x =", x)
print("returned inner =", inner)


outside x = 10
returned inner = 1


### 讲解

输出应是外面 10、返回值 1。若你看见外面变成 1，先检查是否误写了 `global`，或是否在函数外又赋值了一次。验收时能口头指出这一点即可。

### 易错点与练习

把 `print(x)` 放在 `change` 里、放在赋值前，会与“局部 x”规则纠缠，本课不要扩展到那种报错。保持：函数内先赋值再 `return`，函数外再打印两个结果。

1. **K7.1** 调用 `change()` 三次之后，外面的 `x` 是多少？返回值每次是多少？
2. **K7.2** 下面说法哪一句正确：A. 函数改写了外面的 10；B. 函数有自己的 `x`，外面那格没动。

**作答：** 三次调用后外面的 x：____；正确说法：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 8. 组合：默认参数 + 两个返回值 + 拆包

### 理论知识

项目里常要同时交回“个数”和“总和”。例如有效评分的条数与总分，后面再算平均。一次 `return count, total`，调用处拆包。

默认参数可以让评分列下标有一个常用值。本课完整文件评分列是 2；演示用摘录时评分在下标 1，必须显式传入，不能假装默认值到处通用。

### 案例：摘录上统计条数与评分总和


In [8]:
sample = [
    ["人格解体", "8.8", "角色扮演", "TRUE"],
    ["一念逍遥", "6.5", "角色扮演", "TRUE"],
    ["明日方舟", "8.2", "策略", "TRUE"],
    ["另一个伊甸 : 超越时空的猫", "9", "角色扮演", "FALSE"],
    ["偶像梦幻祭2", "8.2", "音乐", "FALSE"],
]
def score_count_total(table, index=1):
    count = 0
    total = 0.0
    for row in table:
        total = total + float(row[index])
        count = count + 1
    return count, total

n, s = score_count_total(sample)
print("count =", n)
print("total =", s)
print("mean =", s / n)
n2, s2 = score_count_total(sample, 1)
print("same call with explicit index:", n2, s2)


count = 5
total = 40.7
mean = 8.14
same call with explicit index: 5 40.7


### 讲解

五条摘录的评分下标是 1，所以默认 `index=1` 只适用于这张简化表。拆包得到条数 5 与总分。平均值用返回值计算，而不是在函数里只 `print`。对完整 `body` 应传 `index=2`，不要把摘录默认值直接套过去。

### 易错点与练习

1. **K8.1** `score_count_total(sample)` 与 `score_count_total(sample, 1)` 是否同一结果？为什么？
2. **K8.2** 对 `body` 调用时若忘记改下标，仍用默认 1，实际加总的是哪一列？

**作答：** 两次调用：____；误用默认下标 1 于 body：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 综合练习：拆包与作用域口答

使用同一份 [mobile_game_info.csv](mobile_game_info.csv)。不要把全表平均分写死。

### P1.1　读取并写出带默认参数的函数

读取文件。写出一个函数：默认抽评分列（下标以表头为准），返回 `(count, total)`。对 `body` 调用一次默认参数版本，再显式传入评分下标各一次。


In [ ]:
# P1.1: Read the file and write a default-parameter function that returns count and total here.


### P1.2　拆包两个返回值

用 `n, s = ...` 拆包。打印条数、总和、平均值。说明返回值的 `type` 在拆包前是什么。

**作答：** 拆包代码：____；拆包前类型：____；平均值如何由 n 与 s 得到：____。


In [ ]:
# P1.2: Unpack the two return values and compute the mean here.


### P1.3　作用域

在笔记本里设置 `x = 10`，定义内部 `x = 1` 并返回内部值的函数。打印外面的 `x` 与返回值。用一句话说明为什么外面仍是 10。不要使用 `global`。

重启内核，从头运行。覆盖 `print` 的实验不要留在最终运行顺序里，或必须在同一格恢复。

**作答：** 外面的 x：____；返回值：____；原因：____。


In [ ]:
# P1.3: Demonstrate inner assignment vs outer x here.


## 本章总结

1. 不要用 `print` / `list` / `str` 给函数起名；演示覆盖后必须恢复内置名。
2. 默认参数让常用下标可以省略。
3. `return a, b` 的类型是 `tuple`；`a, b = fn()` 拆包。
4. 函数里赋值的同名变量不是外面那一格；结果用 `return` 带出，不用 `global`。

带着可复用的读取、抽列、频数函数进入第8课。第8课原则上不再学新语法。

课后请打开 [chapter07_函数进阶_课后练习.ipynb](chapter07_函数进阶_课后练习.ipynb)，使用 [googleplaystore.csv](googleplaystore.csv) 独立完成 P1、P2（P3 选做）。两份数据的列名和列数不同，不能把课堂下标直接套到课后文件上。
